# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [7]:
'''Finding 1: The paper reports that mature pages that were refreshed recently performed better than similarly old pages that remained stale. In the reported local sample, 365+ day content refreshed within 30 days had a 3.2x health-score improvement and substantially higher impressions than the comparison group.

Methodology question:
How was the refresh comparison constructed? In particular, are refreshed and unrefreshed pages comparable in their starting visibility, quality, and other characteristics?

Validation question:
Does the comparison use a design that can separate the observed refresh association from differences that existed before the refresh?

My interpretation:
I would treat this as an observed association and decision-support signal, not proof that refreshing a page caused the improvement.'''

'Finding 1: The paper reports that mature pages that were refreshed recently performed better than similarly old pages that remained stale. In the reported local sample, 365+ day content refreshed within 30 days had a 3.2x health-score improvement and substantially higher impressions than the comparison group.\n\nMethodology question:\nHow was the refresh comparison constructed? In particular, are refreshed and unrefreshed pages comparable in their starting visibility, quality, and other characteristics?\n\nValidation question:\nDoes the comparison use a design that can separate the observed refresh association from differences that existed before the refresh?\n\nMy interpretation:\nI would treat this as an observed association and decision-support signal, not proof that refreshing a page caused the improvement.'

In [8]:
'''Finding 2: The paper reports that its portfolio does not show a simple blanket penalty associated with AI-generated content. Within age-controlled cohorts, stronger and weaker outcomes appeared across different AI-authored groups.

Methodology question:
How was AI authorship or AI workflow defined and measured, and are the compared content groups sufficiently similar in topic, age, quality, and publishing process?

Validation question:
Does comparing model cohorts within age tiers provide enough evidence to generalize the finding to other sites, topics, and publishing workflows?

My interpretation:
I would treat this as a directional observation from the studied portfolio, not as proof that AI-generated content is never penalized.'''

'Finding 2: The paper reports that its portfolio does not show a simple blanket penalty associated with AI-generated content. Within age-controlled cohorts, stronger and weaker outcomes appeared across different AI-authored groups.\n\nMethodology question:\nHow was AI authorship or AI workflow defined and measured, and are the compared content groups sufficiently similar in topic, age, quality, and publishing process?\n\nValidation question:\nDoes comparing model cohorts within age tiers provide enough evidence to generalize the finding to other sites, topics, and publishing workflows?\n\nMy interpretation:\nI would treat this as a directional observation from the studied portfolio, not as proof that AI-generated content is never penalized.'

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [24]:
import pandas as pd

In [20]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

X = model_data_90d[feature_cols_90d]
y = model_data_90d["is_declining"]

print("Rows:", len(X))
print("Features:", feature_cols_90d)
print("Declining rate:", round(y.mean(), 3))

Rows: 87735
Features: ['imp_mid30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'pos_volatility_last30']
Declining rate: 0.656


In [21]:
X_train_old, X_test_old, y_train_old, y_test_old = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

old_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

old_model.fit(X_train_old, y_train_old)

old_pred = old_model.predict(X_test_old)

old_precision = precision_score(y_test_old, old_pred)
old_recall = recall_score(y_test_old, old_pred)
old_f1 = f1_score(y_test_old, old_pred)

print("Original random split")
print("Precision:", round(old_precision, 3))
print("Recall:", round(old_recall, 3))
print("F1:", round(old_f1, 3))

Original random split
Precision: 0.791
Recall: 0.852
F1: 0.82


In [22]:
groups = model_data_90d["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

group_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

group_model.fit(X_train, y_train)

group_pred = group_model.predict(X_test)

group_precision = precision_score(y_test, group_pred)
group_recall = recall_score(y_test, group_pred)
group_f1 = f1_score(y_test, group_pred)

shared_clients = (
    set(groups.iloc[train_idx])
    &
    set(groups.iloc[test_idx])
)

print("Client-grouped split")
print("Precision:", round(group_precision, 3))
print("Recall:", round(group_recall, 3))
print("F1:", round(group_f1, 3))
print("Shared clients:", len(shared_clients))

Client-grouped split
Precision: 0.849
Recall: 0.819
F1: 0.833
Shared clients: 0


In [25]:
comparison = pd.DataFrame({
    "Validation": [
        "Random row split",
        "Client-grouped split"
    ],
    "Precision": [
        old_precision,
        group_precision
    ],
    "Recall": [
        old_recall,
        group_recall
    ],
    "F1": [
        old_f1,
        group_f1
    ]
})

comparison.round(3)

,Validation,Precision,Recall,F1
0,Random row split,0.791,0.852,0.820
1,Client-grouped split,0.849,0.819,0.833


In [26]:
'''The random row split gives an optimistic view of performance because pages from the same client can potentially appear in both training and testing data. I therefore repeated the evaluation using a client-grouped split. The grouped split keeps every client's pages entirely in either training or testing.

I use the grouped result as the more honest estimate of how the model may generalize to unseen clients. If the metrics decrease, that difference is useful evidence that client-specific patterns contributed to the original result.'''

"The random row split gives an optimistic view of performance because pages from the same client can potentially appear in both training and testing data. I therefore repeated the evaluation using a client-grouped split. The grouped split keeps every client's pages entirely in either training or testing.\n\nI use the grouped result as the more honest estimate of how the model may generalize to unseen clients. If the metrics decrease, that difference is useful evidence that client-specific patterns contributed to the original result."

In [9]:
%pip -q install duckdb huggingface_hub

In [10]:

import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [11]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [12]:

clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)

clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [13]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224


In [14]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609,2.0,0.037221,0.823821,34.0,56.0,0.607143
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667,1.0,0.076923,0.446154,93.0,93.0,1.000000
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574,13.0,0.014321,0.718867,226.0,857.0,0.263711
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051,2.0,0.129032,0.760753,22.0,41.0,0.536585
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224,1.0,0.053045,0.843811,105.0,105.0,1.000000


In [15]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))

base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.552     0.343     0.423      9389
           1      0.687     0.839     0.755     16162

    accuracy                          0.656     25551
   macro avg      0.620     0.591     0.589     25551
weighted avg      0.638     0.656     0.633     25551



In [16]:
features_90d = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                        AND f.report_date >  b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_mid30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 60 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 150          -- your custom threshold (was 100 in the 60-day version)
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features_90d):,} content items with enough history (90-day window)')
features_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

95,895 content items with enough history (90-day window)


,client_hash_id,content_hash_id,imp_last30,imp_mid30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,301.0,253.0,0.0,23.401005
1,client_e547b89c05043229,content_48537762b74f5b34,208.0,147.0,396.0,0.0,28.054512
2,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,239.0,185.0,1.0,24.978010
3,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,745.0,714.0,0.0,34.729708
4,client_e547b89c05043229,content_5af024b82aa36744,254.0,159.0,181.0,1.0,8.191330


In [17]:
pos_volatility = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    )
    SELECT f.content_hash_id,
           STDDEV_POP(f.gsc_avg_position) AS pos_volatility_last30
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 30 DAY
    GROUP BY 1
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data_90d = (features_90d
            .merge(qsignals, on='content_hash_id', how='left')
            .merge(pos_volatility, on='content_hash_id', how='left'))

data_90d['is_declining'] = (data_90d['imp_last30'] < 0.8 * data_90d['imp_mid30']).astype(int)

feature_cols_90d = ['imp_mid30', 'visible_queries', 'rare_share', 'anon_share',
                     'top_query_share', 'pos_volatility_last30']
model_data_90d = data_90d.dropna(subset=feature_cols_90d)

print(f'joined: {len(data_90d):,} rows')
data_90d.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 95,895 rows


,client_hash_id,content_hash_id,imp_last30,imp_mid30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,pos_volatility_last30,is_declining
0,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,301.0,253.0,0.0,23.401005,4.0,0.130841,0.562083,121.0,230.0,0.526087,9.206046,1
1,client_e547b89c05043229,content_48537762b74f5b34,208.0,147.0,396.0,0.0,28.054512,1.0,0.169108,0.797603,25.0,25.0,1.000000,17.180979,0
2,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,239.0,185.0,1.0,24.978010,5.0,0.186380,0.620072,33.0,108.0,0.305556,8.802450,1
3,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,745.0,714.0,0.0,34.729708,12.0,0.056039,0.817193,37.0,233.0,0.158798,13.434683,1
4,client_e547b89c05043229,content_5af024b82aa36744,254.0,159.0,181.0,1.0,8.191330,4.0,0.104377,0.789562,24.0,63.0,0.380952,6.861256,0


In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

data_90d['is_declining'] = (data_90d['imp_last30'] < 0.8 * data_90d['imp_mid30']).astype(int)

feature_cols_90d = ['imp_mid30', 'visible_queries', 'rare_share', 'anon_share',
                     'top_query_share', 'pos_volatility_last30']
model_data_90d = data_90d.dropna(subset=feature_cols_90d)

X = model_data_90d[feature_cols_90d]
y = model_data_90d['is_declining']
groups = model_data_90d['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# sanity check: no client should appear in both splits
assert set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]) == set()

model_grouped = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'clients in train: {groups.iloc[train_idx].nunique()}, clients in test: {groups.iloc[test_idx].nunique()}')
print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model_grouped.predict(X_te), digits=3))

clients in train: 31, clients in test: 11
base rate (always predict majority): 0.701
              precision    recall  f1-score   support

           0      0.607     0.658     0.632      2693
           1      0.849     0.819     0.833      6320

    accuracy                          0.771      9013
   macro avg      0.728     0.738     0.733      9013
weighted avg      0.777     0.771     0.773      9013



## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [27]:
dangerous_features = [
    "is_declining",
    "trend_direction",
    "trend_pct"
]

print("Leakage audit")

for feature in dangerous_features:
    print(
        feature,
        "USED AS FEATURE:" if feature in feature_cols_90d else "NOT USED AS FEATURE"
    )

Leakage audit
is_declining NOT USED AS FEATURE
trend_direction NOT USED AS FEATURE
trend_pct NOT USED AS FEATURE


In [28]:
print("Features used by the model:")

for feature in feature_cols_90d:
    print("-", feature)

Features used by the model:
- imp_mid30
- visible_queries
- rare_share
- anon_share
- top_query_share
- pos_volatility_last30


In [29]:
feature_timing = pd.DataFrame({
    "Feature": feature_cols_90d,
    "Available before decision": [
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ],
    "Why": [
        "Uses historical impressions from the middle 30-day period",
        "Uses observed query visibility",
        "Uses historical rare-query share",
        "Uses historical anonymized-query share",
        "Uses historical query impression concentration",
        "Uses historical position volatility"
    ]
})

feature_timing

,Feature,Available before decision,Why
0,imp_mid30,Yes,Uses historical impressions from the middle 30...
1,visible_queries,Yes,Uses observed query visibility
2,rare_share,Yes,Uses historical rare-query share
3,anon_share,Yes,Uses historical anonymized-query share
4,top_query_share,Yes,Uses historical query impression concentration
5,pos_volatility_last30,Yes,Uses historical position volatility


In [30]:
'''I checked the model features for direct label leakage. The target is_declining and the source fields trend_direction and trend_pct are not used as model features. These fields are directly related to the outcome definition and would leak future or label-derived information into the model.

The selected features are historical search and query signals intended to be available before the content-review decision.'''

'I checked the model features for direct label leakage. The target is_declining and the source fields trend_direction and trend_pct are not used as model features. These fields are directly related to the outcome definition and would leak future or label-derived information into the model.\n\nThe selected features are historical search and query signals intended to be available before the content-review decision.'

In [32]:
test_results = model_data_90d.iloc[test_idx].copy()

test_results["actual"] = y_test.values
test_results["predicted"] = group_pred

test_results["error_type"] = "Correct"

test_results.loc[
    (test_results["actual"] == 1) &
    (test_results["predicted"] == 0),
    "error_type"
] = "False Negative"

test_results.loc[
    (test_results["actual"] == 0) &
    (test_results["predicted"] == 1),
    "error_type"
] = "False Positive"

errors = test_results[
    test_results["error_type"] != "Correct"
]

print("Total errors:", len(errors))

errors[
    feature_cols_90d +
    ["actual", "predicted", "error_type"]
].head(10)

Total errors: 2067


,imp_mid30,visible_queries,rare_share,anon_share,top_query_share,pos_volatility_last30,actual,predicted,error_type
4061,2701.0,104.0,0.113652,0.071077,0.167458,4.614860,0,1,False Positive
4069,85.0,1.0,0.086111,0.883333,1.000000,9.618529,1,0,False Negative
4075,3004.0,49.0,0.054972,0.730957,0.058738,3.833324,0,1,False Positive
4076,99.0,1.0,0.124760,0.850288,1.000000,21.263586,1,0,False Negative
4078,174.0,6.0,0.290230,0.341954,0.460938,7.930252,1,0,False Negative
4079,201.0,6.0,0.232906,0.551282,0.445545,6.763497,1,0,False Negative
4093,282.0,2.0,0.164062,0.796875,0.520000,1.021376,1,0,False Negative
4102,128.0,11.0,0.234628,0.328479,0.307407,18.724275,1,0,False Negative
4109,205.0,5.0,0.127273,0.745455,0.233766,5.318432,1,0,False Negative
4112,80.0,2.0,0.136591,0.832080,0.520000,6.086930,1,0,False Negative


In [34]:
'''The model makes both false-positive and false-negative errors. A false positive means the model would prioritize a page that does not meet the defined declining condition. A false negative means the model misses a page that does meet the declining condition.

These errors show why the model should be used as decision support for content review rather than as an automatic decision-maker.'''

'The model makes both false-positive and false-negative errors. A false positive means the model would prioritize a page that does not meet the defined declining condition. A false negative means the model misses a page that does meet the declining condition.\n\nThese errors show why the model should be used as decision support for content review rather than as an automatic decision-maker.'

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [33]:
'''Original claim:
The model predicts which pages will decline.

More careful claim:
The model ranks pages according to their association with the defined declining-performance label and can support content-review prioritization.

Original claim:
The model identifies what causes traffic decline.

More careful claim:
The model identifies features that are useful for distinguishing pages with the defined declining-performance pattern. This does not establish causation.

Original claim:
The model predicts Google's algorithm.

More careful claim:
The model does not predict Google's algorithm. It provides decision support using the search and content signals available in this dataset.

Original claim:
Refreshing content causes performance to improve.

More careful claim:
The research paper reports an observed association between recent refreshing and stronger performance among certain mature pages, but the observational design does not establish causation.'''

"Original claim:\nThe model predicts which pages will decline.\n\nMore careful claim:\nThe model ranks pages according to their association with the defined declining-performance label and can support content-review prioritization.\n\nOriginal claim:\nThe model identifies what causes traffic decline.\n\nMore careful claim:\nThe model identifies features that are useful for distinguishing pages with the defined declining-performance pattern. This does not establish causation.\n\nOriginal claim:\nThe model predicts Google's algorithm.\n\nMore careful claim:\nThe model does not predict Google's algorithm. It provides decision support using the search and content signals available in this dataset.\n\nOriginal claim:\nRefreshing content causes performance to improve.\n\nMore careful claim:\nThe research paper reports an observed association between recent refreshing and stronger performance among certain mature pages, but the observational design does not establish causation."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.